# Advanced RAG — Production Techniques

**Phase 04 · Improving the Baseline Pipeline**

---

## ⚠️ Prerequisites

| Requirement | How to satisfy |
|---|---|
| Phase 03 complete | Run `rag_pipeline.ipynb` first to generate `support_tickets.csv` |
| **Ollama running** | `ollama serve` in a terminal |
| **llama3 pulled** | `ollama pull llama3` |
| **Python packages** | `pip install langchain langchain-community chromadb sentence-transformers rank-bm25 ragas pandas faker requests` |

All models are **100% local** — no API keys, no cloud costs.

---

**By the end of this notebook you will be able to:**

- Measure RAG quality objectively with RAGAS metrics
- Improve retrieval precision with parent-child chunking
- Improve answer quality with cross-encoder reranking
- Handle keyword queries with BM25 + vector hybrid search
- Show a before/after comparison table that proves which techniques helped


In [ ]:
import os, json, random, shutil, math, warnings
import pandas as pd
import requests
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = os.path.abspath('')
DATA_DIR     = os.path.join(NOTEBOOK_DIR, '..', '..', 'data', 'raw')
TICKETS_PATH = os.path.join(DATA_DIR, 'support_tickets.csv')

# ── Auto-generate support_tickets.csv if missing ──────────────────────────
if not os.path.exists(TICKETS_PATH):
    print('support_tickets.csv not found — generating...')
    try:
        from faker import Faker
        fake = Faker()
    except ImportError:
        raise SystemExit('Run: pip install faker')

    HIGH = [
        ('Production database is down',
         'Our entire production database cluster has been unreachable for 30 minutes. '
         'All customers are affected. Revenue impact is severe. '
         'Engineers are on-call but need platform access restored urgently.'),
        ('Critical security breach detected',
         'We have detected unauthorised access to customer PII. '
         'The attack appears ongoing. Immediate escalation required. '
         'All API keys for the affected service should be rotated immediately.'),
        ('Payment processing completely broken',
         'No payments are going through. Checkout is broken for 100% of users. '
         'We are losing revenue every minute. '
         'Last deployment was 2 hours ago and rollback has not fixed the issue.'),
        ('SSL certificate expired — site unreachable',
         'Our SSL cert expired 2 hours ago. All users see a browser security warning. '
         'The site is effectively offline. Auto-renewal did not trigger.'),
        ('Authentication service returning 500 errors',
         'Users cannot log in. Auth service has been throwing 500s since the '
         'last deployment 45 minutes ago. The rollback procedure has not resolved it.'),
        ('Data corruption in billing records',
         'Invoice amounts are incorrect for approximately 2,000 customers. '
         'The monthly billing cycle runs in 6 hours. '
         'Root cause appears to be a bad migration script applied yesterday.'),
        ('All backups missing from last 7 days',
         'Scheduled backup job silently failed for a week. '
         'We have no valid recovery point. Risk of permanent data loss is high.'),
        ('API rate limits not enforced — data leak risk',
         'Our public API has stopped enforcing rate limits. '
         'Malicious actors are actively scraping all customer data at high volume.'),
    ]
    MEDIUM = [
        ('Dashboard export generates wrong totals',
         'The CSV export from the analytics dashboard shows figures that do not '
         'match the on-screen numbers. Discrepancies are around 5%. '
         'Affects finance team monthly reporting.'),
        ('Email notifications delayed by 4+ hours',
         'Users report that welcome and password-reset emails arrive hours late. '
         'This is causing increased support calls from users locked out of accounts.'),
        ('Search returns irrelevant results after update',
         'Since last Monday\'s release, the search bar frequently returns unrelated '
         'items. A query for "invoice" returns blog posts. Users are complaining.'),
        ('Mobile app crashes on iOS 17.4',
         'Several users report the iOS app crashes immediately on launch after '
         'upgrading to iOS 17.4. The issue is reproducible. Android is unaffected.'),
        ('Charts not rendering in Firefox',
         'All bar and line charts are blank in Firefox 124+. '
         'Charts renders correctly in Chrome and Safari. Affects roughly 15% of users.'),
        ('Bulk import fails on files larger than 5 MB',
         'The CSV import wizard silently fails for files over 5 MB '
         'without showing an error. Users lose work without knowing why.'),
        ('Report generation taking over 10 minutes',
         'Monthly summary reports that used to generate in 30 seconds '
         'now take over 10 minutes. Database query plan may have degraded.'),
        ('Wrong timezone applied to scheduled tasks',
         'Scheduled tasks run in UTC rather than the configured user timezone. '
         'This causes tasks to fire 1-8 hours late depending on user location.'),
    ]
    LOW = [
        ('Request to add dark mode',
         'Many users have requested a dark mode theme option. '
         'Would be nice to have for evening use. No business impact.'),
        ('Typo on pricing page',
         'The word "anually" should be "annually" on the /pricing page. Minor issue.'),
        ('Tooltip text is hard to read',
         'The tooltip on the settings page uses light grey text on white background. '
         'Accessibility concern but low impact overall.'),
        ('Add keyboard shortcut for save',
         'It would be helpful to have Ctrl+S / Cmd+S save the current form. '
         'Power users would appreciate this quality-of-life improvement.'),
        ('Docs link broken on help page',
         'The Getting Started link in the Help section returns a 404. '
         'Should be fixed in the next content deploy.'),
        ('Export button should confirm before running',
         'Users accidentally trigger large exports. '
         'A confirmation dialog would prevent accidental heavy jobs.'),
        ('Date picker does not support manual entry',
         'The date picker widget requires clicking. '
         'Users who prefer typing cannot enter dates directly.'),
        ('Add option to change default currency',
         'The UI always defaults to USD. '
         'Allow users to set a default display currency in profile settings.'),
    ]

    random.seed(42)
    rows = []
    ticket_id = 1001
    for items, priority in [(HIGH, 'high'), (MEDIUM, 'medium'), (LOW, 'low')]:
        for subject, body in items * 5:
            rows.append({'id': ticket_id, 'subject': subject,
                         'body': body, 'priority': priority,
                         'created_by': fake.name()})
            ticket_id += 1
    random.shuffle(rows)
    os.makedirs(DATA_DIR, exist_ok=True)
    pd.DataFrame(rows).to_csv(TICKETS_PATH, index=False)
    print(f'Generated {len(rows)} tickets -> {TICKETS_PATH}')

df = pd.read_csv(TICKETS_PATH)
print(f'Loaded {len(df)} tickets | columns: {list(df.columns)}')

# ── Helpers ───────────────────────────────────────────────────────────────
OLLAMA_URL = 'http://localhost:11434/api/chat'
MODEL      = 'llama3'

def ollama_chat(messages: list[dict], temperature: float = 0.1) -> str:
    try:
        r = requests.post(
            OLLAMA_URL,
            json={'model': MODEL, 'messages': messages,
                  'stream': False, 'options': {'temperature': temperature}},
            timeout=120,
        )
        r.raise_for_status()
        return r.json()['message']['content']
    except requests.exceptions.ConnectionError:
        return '[ERROR] Cannot reach Ollama. Ensure `ollama serve` is running.'


def ollama_generate(prompt: str, temperature: float = 0.1) -> str:
    return ollama_chat([{'role': 'user', 'content': prompt}], temperature)


print('Setup complete.')

---

## Section 1 — Baseline Measurement with RAGAS

### You can't improve what you can't measure

The basic RAG pipeline from Phase 03 *seems* to work — answers look reasonable.
But "looks reasonable" is not engineering. Before making changes we need a
**numeric baseline** so we can tell, objectively, whether any change improved things.

### RAGAS metrics

RAGAS (Retrieval-Augmented Generation Assessment) evaluates RAG pipelines without
needing human judges. It uses an LLM to score each response:

| Metric | What it measures | Good score means... |
|--------|-----------------|--------------------|
| **Faithfulness** | Does the answer stick to the retrieved context? | Model does not hallucinate beyond the docs |
| **Answer relevancy** | Does the answer address the question? | Model answers what was asked |
| **Context recall** | Did the retrieved docs contain the answer? | Retrieval found the right chunks |

All three metrics return a score between 0 and 1. Higher is better.

### Our approach

RAGAS needs a test set of `(question, ground_truth_answer, retrieved_contexts, generated_answer)`
tuples. We:

1. Write 15 question-answer pairs manually from our ticket data
2. Run each question through the Phase 03 basic RAG pipeline
3. Feed the results to RAGAS
4. Save scores as `baseline_scores`

### Why Ollama as the RAGAS judge?

RAGAS normally calls OpenAI. We patch it to use our local Ollama endpoint instead —
free, private, no API key required.

### What to look for
- Baseline faithfulness < 0.7 → model is often adding info not in retrieved docs
- Low context recall → wrong chunks being retrieved
- These numbers are the target to beat in Sections 2–4

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# ── Build basic RAG pipeline (same as Phase 03) ───────────────────────────
BASIC_CHROMA_DIR = os.path.join(NOTEBOOK_DIR, 'chroma_basic')
if os.path.exists(BASIC_CHROMA_DIR):
    shutil.rmtree(BASIC_CHROMA_DIR)

docs_raw = [
    f"[Ticket {row['id']} | Priority: {row['priority'].upper()}]\n"
    f"Subject: {row['subject']}\nBody: {row['body']}"
    for _, row in df.iterrows()
]

basic_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', ''],
)
basic_chunks = basic_splitter.create_documents(
    texts=docs_raw,
    metadatas=[{'ticket_id': int(row['id']), 'priority': row['priority']}
               for _, row in df.iterrows()],
)

print('Loading embedding model...')
embedding_fn = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print(f'Indexing {len(basic_chunks)} chunks into basic Chroma...')
basic_vs = Chroma.from_documents(
    documents=basic_chunks,
    embedding=embedding_fn,
    persist_directory=BASIC_CHROMA_DIR,
)
print(f'Basic vector store ready — {basic_vs._collection.count()} chunks')

In [ ]:
# ── 15-question test set (hand-curated from ticket data) ─────────────────
TEST_SET = [
    # --- high priority ---
    {
        'question': 'What is the status of the production database?',
        'ground_truth': 'The production database cluster has been unreachable for 30 minutes affecting all customers.',
    },
    {
        'question': 'Why are users unable to log in?',
        'ground_truth': 'The authentication service has been returning 500 errors since the last deployment 45 minutes ago.',
    },
    {
        'question': 'What happened to the SSL certificate?',
        'ground_truth': 'The SSL certificate expired and auto-renewal did not trigger, making the site unreachable.',
    },
    {
        'question': 'Is there a security breach in progress?',
        'ground_truth': 'Yes, unauthorised access to customer PII has been detected and the attack appears ongoing.',
    },
    {
        'question': 'What is wrong with payments?',
        'ground_truth': 'No payments are going through; checkout is broken for all users and rollback has not fixed it.',
    },
    # --- medium priority ---
    {
        'question': 'Why are email notifications delayed?',
        'ground_truth': 'Welcome and password-reset emails are arriving hours late, causing users to be locked out.',
    },
    {
        'question': 'Which browsers are affected by the charts rendering bug?',
        'ground_truth': 'Charts are blank in Firefox 124+; Chrome and Safari are unaffected.',
    },
    {
        'question': 'What is causing incorrect report totals in the dashboard?',
        'ground_truth': 'The CSV export shows figures that do not match on-screen numbers with discrepancies around 5%.',
    },
    {
        'question': 'Why is the mobile app crashing?',
        'ground_truth': 'The iOS app crashes on launch after upgrading to iOS 17.4; Android is unaffected.',
    },
    {
        'question': 'What happens when bulk imports exceed 5 MB?',
        'ground_truth': 'The CSV import wizard silently fails for files over 5 MB without showing an error.',
    },
    # --- low priority ---
    {
        'question': 'What feature request is most commonly mentioned?',
        'ground_truth': 'Many users have requested a dark mode theme option.',
    },
    {
        'question': 'Is there a typo reported anywhere on the site?',
        'ground_truth': 'Yes, the word "anually" should be "annually" on the /pricing page.',
    },
    {
        'question': 'What accessibility issue has been raised?',
        'ground_truth': 'The tooltip uses light grey text on white background, making it hard to read.',
    },
    {
        'question': 'Is there a broken link in the help section?',
        'ground_truth': 'The Getting Started link in the Help section returns a 404.',
    },
    {
        'question': 'What keyboard shortcut has been requested?',
        'ground_truth': 'Users have requested Ctrl+S / Cmd+S as a keyboard shortcut for saving the current form.',
    },
]

print(f'Test set: {len(TEST_SET)} question-answer pairs')

In [ ]:
# ── Basic RAG pipeline function ───────────────────────────────────────────
RAG_SYSTEM = (
    'You are a helpful support desk assistant. '
    'Answer the question using ONLY the context provided below. '
    'If the answer cannot be found in the context, '
    'respond with exactly: "I don\'t have that information." '
    'Be concise — 2 to 4 sentences maximum.'
)

def basic_rag(question: str, k: int = 3) -> dict:
    """Run basic RAG, return dict with answer + contexts."""
    docs_scores = basic_vs.similarity_search_with_score(question, k=k)
    contexts = [doc.page_content for doc, _ in docs_scores]
    context_block = '\n\n'.join(
        f'[Source {i+1}]\n{c}' for i, c in enumerate(contexts)
    )
    answer = ollama_chat([
        {'role': 'system', 'content': RAG_SYSTEM},
        {'role': 'user',   'content': f'CONTEXT:\n{context_block}\n\nQUESTION: {question}'},
    ])
    return {'answer': answer, 'contexts': contexts}


# ── Run test set through basic RAG ────────────────────────────────────────
print('Running 15 questions through basic RAG pipeline...')
print('(This may take 2–10 minutes depending on your hardware)\n')

basic_results = []
for i, item in enumerate(TEST_SET, 1):
    result = basic_rag(item['question'])
    basic_results.append({
        'question':      item['question'],
        'ground_truth':  item['ground_truth'],
        'answer':        result['answer'],
        'contexts':      result['contexts'],
    })
    print(f'  [{i:02d}/15] Q: {item["question"][:60]}...')
    print(f'         A: {result["answer"][:80]}...')

print('\nBasic RAG inference complete.')

In [ ]:
# ── RAGAS evaluation using Ollama as the judge ───────────────────────────
# RAGAS >= 0.2 uses a langchain_core LLM wrapper. We build a minimal one
# that routes to our local Ollama endpoint.
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.llms import Ollama as LangchainOllama
from datasets import Dataset

# Local Ollama LLM wrapper for RAGAS
ragas_llm = LangchainLLMWrapper(LangchainOllama(model=MODEL, base_url='http://localhost:11434'))
ragas_emb = LangchainEmbeddingsWrapper(embedding_fn)

# Build HuggingFace Dataset that RAGAS expects
ragas_data = {
    'question':   [r['question']    for r in basic_results],
    'answer':     [r['answer']      for r in basic_results],
    'contexts':   [r['contexts']    for r in basic_results],
    'ground_truth': [r['ground_truth'] for r in basic_results],
}
basic_dataset = Dataset.from_dict(ragas_data)

print('Running RAGAS evaluation on basic pipeline...')
print('(Uses Ollama as judge — expect 5–15 minutes)\n')

metrics = [faithfulness, answer_relevancy, context_recall]
for m in metrics:
    m.llm = ragas_llm
    if hasattr(m, 'embeddings'):
        m.embeddings = ragas_emb

basic_eval = evaluate(
    dataset=basic_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_emb,
    raise_exceptions=False,
)

baseline_scores = {
    'faithfulness':     round(float(basic_eval['faithfulness']), 3),
    'answer_relevancy': round(float(basic_eval['answer_relevancy']), 3),
    'context_recall':   round(float(basic_eval['context_recall']), 3),
}

print('\n── BASELINE SCORES ──')
for k, v in baseline_scores.items():
    bar = '█' * int(v * 20)
    print(f'  {k:<20} {v:.3f}  {bar}')

---

## Section 2 — Better Chunking: Parent-Child

### The retrieval precision problem

Larger chunks contain more context but are harder to match precisely.
Smaller chunks match queries more precisely but may lack surrounding context
needed for a complete answer.

**Parent-child chunking** solves the tension:

```
Parent chunk (300 chars) — full context, what we send to the LLM
       │
       ├── child chunk A (150 chars) — indexed in vector store
       ├── child chunk B (150 chars) — indexed in vector store
       └── child chunk C (150 chars) — indexed in vector store
```

**Retrieval steps:**
1. Embed the query and search only against **child** chunks (precise match)
2. Look up the **parent** chunk that contains the matched child
3. Send the full **parent** context to the LLM

### Why this helps

| | Standard chunking | Parent-child |
|--|---|---|
| Retrieval target | Chunk (300 chars) | Child (150 chars) — more specific |
| Context sent to LLM | Same chunk | Parent (300 chars) — full context |
| Match precision | Medium | High |
| Context completeness | Medium | High |

### What to look for
- Retrieved parent chunk is larger and contains the answer more completely
- Context recall score should improve because the right chunk surfaces more reliably

In [ ]:
# ── Build parent-child index ──────────────────────────────────────────────
PC_CHROMA_DIR = os.path.join(NOTEBOOK_DIR, 'chroma_parent_child')
if os.path.exists(PC_CHROMA_DIR):
    shutil.rmtree(PC_CHROMA_DIR)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', ''],
)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150, chunk_overlap=20,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

# Build parent chunks and a {child_id -> parent_text} mapping
parent_chunks = parent_splitter.create_documents(
    texts=docs_raw,
    metadatas=[{'ticket_id': int(row['id']), 'priority': row['priority']}
               for _, row in df.iterrows()],
)

child_docs   = []   # LangChain Documents (for Chroma)
parent_store = {}  # {child_id: parent_text}

for p_idx, parent in enumerate(parent_chunks):
    children = child_splitter.split_documents([parent])
    for c_idx, child in enumerate(children):
        child_id = f'p{p_idx}_c{c_idx}'
        child.metadata['child_id']    = child_id
        child.metadata['parent_idx']  = p_idx
        parent_store[child_id]        = parent.page_content
        child_docs.append(child)

print(f'Parent chunks : {len(parent_chunks)}')
print(f'Child chunks  : {len(child_docs)}')
print(f'Ratio         : {len(child_docs)/len(parent_chunks):.1f} children per parent')

pc_vs = Chroma.from_documents(
    documents=child_docs,
    embedding=embedding_fn,
    persist_directory=PC_CHROMA_DIR,
)
print(f'\nParent-child vector store ready — {pc_vs._collection.count()} child chunks indexed')

print('\nExample:')
print('Child chunk:', child_docs[0].page_content)
print('\nParent chunk:', parent_store[child_docs[0].metadata['child_id']])

In [ ]:
def parent_child_rag(question: str, k: int = 3) -> dict:
    """Retrieve child chunks, look up parent, generate from parent context."""
    child_hits = pc_vs.similarity_search_with_score(question, k=k)

    # De-duplicate parents (multiple children can share a parent)
    seen_parents = {}
    for doc, score in child_hits:
        cid = doc.metadata['child_id']
        parent_text = parent_store[cid]
        p_idx = doc.metadata['parent_idx']
        if p_idx not in seen_parents:
            seen_parents[p_idx] = (parent_text, score)

    contexts = [text for text, _ in seen_parents.values()]
    context_block = '\n\n'.join(
        f'[Source {i+1}]\n{c}' for i, c in enumerate(contexts)
    )
    answer = ollama_chat([
        {'role': 'system', 'content': RAG_SYSTEM},
        {'role': 'user',   'content': f'CONTEXT:\n{context_block}\n\nQUESTION: {question}'},
    ])
    return {'answer': answer, 'contexts': contexts}


# ── Compare 5 answers: basic vs parent-child ──────────────────────────────
compare_qs = [
    'What is causing users to be unable to log in?',
    'What happened to the SSL certificate?',
    'Why are email notifications delayed?',
    'Which browsers are affected by the charts rendering bug?',
    'What accessibility issue has been raised?',
]

print('BASIC RAG  vs  PARENT-CHILD RAG')
print('=' * 70)
for q in compare_qs:
    basic_ans = basic_rag(q)['answer']
    pc_ans    = parent_child_rag(q)['answer']
    print(f'\nQ: {q}')
    print(f'Basic   : {basic_ans[:120]}...')
    print(f'Parent  : {pc_ans[:120]}...')

---

## Section 3 — Reranking with CrossEncoder

### Bi-encoder vs cross-encoder

The `all-MiniLM-L6-v2` model we use for embeddings is a **bi-encoder**:

```
Bi-encoder (fast, approximate)
  query  ──► encoder ──► vector_q
  chunk  ──► encoder ──► vector_c
  similarity = cosine(vector_q, vector_c)
```

It is fast because each chunk is encoded independently. But because
query and chunk never interact during encoding, it misses subtle relationships.

A **cross-encoder** processes the (query, chunk) pair together:

```
Cross-encoder (slow, accurate)
  [query] [SEP] [chunk]  ──►  transformer  ──►  relevance score
```

The two tokens attend to each other through every transformer layer.
This captures much richer relevance signals — but it cannot be pre-computed
and must run at query time for every (query, candidate) pair.

### Two-stage retrieval

```
Query
  │
  ▼
Stage 1: bi-encoder similarity search  →  top 20 candidates  (fast)
  │
  ▼
Stage 2: cross-encoder rerank         →  top 5 results       (accurate)
  │
  ▼
LLM generation with top-5 context
```

We use **`cross-encoder/ms-marco-MiniLM-L-6-v2`** — a 22 MB model trained on
the MS MARCO passage ranking dataset, free from Hugging Face.

### What to look for
- Stage 1 candidate order often differs from Stage 2 reranked order
- The model that was ranked 8th can jump to rank 1 after reranking
- Faithfulness score should improve because the right chunks consistently surface

In [ ]:
from sentence_transformers import CrossEncoder

print('Loading CrossEncoder (ms-marco-MiniLM-L-6-v2)...')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
print('CrossEncoder ready.')


def reranked_rag(question: str, first_stage_k: int = 20, final_k: int = 5) -> dict:
    """Two-stage retrieve-then-rerank RAG."""
    # Stage 1: fast bi-encoder retrieval
    candidates = basic_vs.similarity_search(question, k=first_stage_k)

    # Stage 2: cross-encoder rerank
    pairs  = [(question, doc.page_content) for doc in candidates]
    scores = cross_encoder.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True,
    )
    top_docs = [doc for doc, _ in ranked[:final_k]]

    contexts = [d.page_content for d in top_docs]
    context_block = '\n\n'.join(
        f'[Source {i+1}]\n{c}' for i, c in enumerate(contexts)
    )
    answer = ollama_chat([
        {'role': 'system', 'content': RAG_SYSTEM},
        {'role': 'user',   'content': f'CONTEXT:\n{context_block}\n\nQUESTION: {question}'},
    ])
    return {'answer': answer, 'contexts': contexts, 'rerank_scores': [s for _, s in ranked[:final_k]]}


# ── Demonstrate reranking shift ────────────────────────────────────────────
demo_q = 'Why are users unable to log in?'
stage1 = basic_vs.similarity_search(demo_q, k=10)
pairs  = [(demo_q, d.page_content) for d in stage1]
ce_scores = cross_encoder.predict(pairs)

print(f'Demo query: "{demo_q}"\n')
print('Stage 1 order (bi-encoder) vs Stage 2 rerank (cross-encoder):')
print(f'{"Rank":>5}  {"CE Score":>9}  Snippet')
print('─' * 70)
for orig_rank, (doc, ce_s) in enumerate(
    sorted(zip(stage1, ce_scores), key=lambda x: x[1], reverse=True), 1
):
    snippet = doc.page_content[:60].replace('\n', ' ')
    print(f'{orig_rank:>5}  {ce_s:>9.3f}  {snippet}...')

print('\n── Running reranked RAG on 5 comparison queries ──')
for q in compare_qs:
    result = reranked_rag(q)
    print(f'\nQ: {q}')
    print(f'Reranked: {result["answer"][:120]}...')
    print(f'CE scores (top 5): {[round(s,2) for s in result["rerank_scores"]]}')

---

## Section 4 — Hybrid Search: BM25 + Vector

### The keyword gap in vector search

Vector search finds *semantically similar* text. But it can miss text that is
**exactly identical** to the query — especially for:

- **Ticket IDs** — "ticket 1023" → semantic search may return tickets about similar topics, not 1023
- **Proper names** — "John Smith" → similar embeddings to other names
- **Error codes** — "HTTP 500", "ERR_SSL_PROTOCOL_ERROR"
- **Acronyms** — "PII", "API", "SSL"

**BM25** (Best Match 25) is a classic keyword ranking function. It scores
documents by term frequency and inverse document frequency — pure lexical overlap.
No embeddings, no neural network, no GPU required.

### Reciprocal Rank Fusion (RRF)

We cannot directly average BM25 scores and cosine distances because they are
on different scales. **RRF** fuses ranked lists:

```
rrf_score(doc) = Σ 1 / (k + rank_in_list_i)

  k = 60 (smoothing constant, reduces the advantage of very high ranks)
```

Higher `k` makes lower-ranked items more competitive. `k=60` is the standard
choice from the original RRF paper.

### Pipeline

```
Query
  │
  ├──► BM25 search    → ranked list 1
  │                          │
  ├──► Vector search  → ranked list 2
  │                          │
  └──────────────────► RRF fusion → combined top-k
                                        │
                                        ▼
                                   LLM generation
```

### What to look for
- Queries with exact ticket numbers, names, or codes surface the correct document
- Pure vector search misses these; hybrid search finds them
- Context recall score should improve for keyword-heavy queries

In [ ]:
from rank_bm25 import BM25Okapi

# ── Build BM25 index on the same basic chunks ─────────────────────────────
corpus_texts = [c.page_content for c in basic_chunks]

def tokenize(text: str) -> list[str]:
    """Simple whitespace + lowercase tokenizer."""
    return text.lower().split()

tokenized_corpus = [tokenize(t) for t in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)
print(f'BM25 index built over {len(corpus_texts)} chunks.')


def rrf_score(rankings: list[list[int]], k: int = 60) -> dict[int, float]:
    """
    Compute RRF score for each document index.
    rankings: list of lists, each list is an ordered sequence of doc indices.
    """
    scores: dict[int, float] = {}
    for ranked_list in rankings:
        for rank, doc_idx in enumerate(ranked_list, start=1):
            scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)
    return scores


def hybrid_search(query: str, k: int = 10) -> list[str]:
    """
    Combine BM25 + vector search via RRF, return top-k chunk texts.
    """
    # BM25 ranked list
    bm25_scores  = bm25.get_scores(tokenize(query))
    bm25_ranking = list(bm25_scores.argsort()[::-1][:50])   # top 50

    # Vector search ranked list
    vec_results  = basic_vs.similarity_search(query, k=50)
    # Match vector results back to corpus_texts indices
    text_to_idx  = {t: i for i, t in enumerate(corpus_texts)}
    vec_ranking  = [
        text_to_idx[doc.page_content]
        for doc in vec_results
        if doc.page_content in text_to_idx
    ]

    # RRF fusion
    fused = rrf_score([bm25_ranking, vec_ranking], k=60)
    top_indices = sorted(fused, key=fused.get, reverse=True)[:k]

    return [corpus_texts[i] for i in top_indices]


def hybrid_rag(question: str, k: int = 5) -> dict:
    """Full RAG using hybrid BM25+vector retrieval."""
    contexts = hybrid_search(question, k=k)
    context_block = '\n\n'.join(
        f'[Source {i+1}]\n{c}' for i, c in enumerate(contexts)
    )
    answer = ollama_chat([
        {'role': 'system', 'content': RAG_SYSTEM},
        {'role': 'user',   'content': f'CONTEXT:\n{context_block}\n\nQUESTION: {question}'},
    ])
    return {'answer': answer, 'contexts': contexts}


# ── Test on 5 keyword-critical queries ────────────────────────────────────
keyword_queries = [
    'What happened with the SSL certificate?',         # acronym: SSL
    'Is there a ticket about iOS 17.4?',               # exact version number
    'Is customer PII at risk?',                        # acronym: PII
    'Are there any HTTP 500 errors reported?',         # error code
    'Is there a Firefox-specific bug?',                # proper noun
]

print('HYBRID SEARCH TEST — keyword-critical queries')
print('=' * 70)
for q in keyword_queries:
    basic_ans  = basic_rag(q)['answer']
    hybrid_ans = hybrid_rag(q)['answer']
    print(f'\nQ: {q}')
    print(f'  Basic   : {basic_ans[:100]}...')
    print(f'  Hybrid  : {hybrid_ans[:100]}...')

---

## Section 5 — Final Measurement

### Always re-evaluate after changes

Every technique in this notebook was motivated by a hypothesis:

- Parent-child → better context recall
- Reranking → better faithfulness
- Hybrid search → better retrieval on keyword queries

Now we test those hypotheses with RAGAS. We build a combined "improved" pipeline
that uses all three techniques together and evaluate it against the same 15-question
test set.

### The improved pipeline

```
Query
  │
  ├── Hybrid search (BM25 + vector, top 30)
  │         └── retrieve child chunks → look up parents
  │
  ▼
  Cross-encoder rerank (top 30 → top 5)
  │
  ▼
  Ollama generation with top-5 parent contexts
```

### What to look for
- Context recall should be highest (hybrid + parent-child)
- Faithfulness should be highest (reranking surfaces most relevant chunks)
- The comparison table shows which metric benefited most

In [ ]:
# ── Build improved pipeline (hybrid + parent-child + reranking) ──────────

# BM25 on child chunks for more precise lexical matching
child_texts      = [c.page_content for c in child_docs]
child_bm25       = BM25Okapi([tokenize(t) for t in child_texts])
child_text_to_id = {c.page_content: c.metadata['child_id'] for c in child_docs}


def improved_rag(question: str,
                 first_stage_k: int = 30,
                 final_k: int = 5) -> dict:
    """
    Improved pipeline:
    1. Hybrid search on child chunks (BM25 + vector)
    2. Look up parent chunks
    3. Cross-encoder rerank parents
    4. Generate from top-final_k parents
    """
    # ── Stage 1a: BM25 over child chunks ──────────────────────────────────
    bm25_scores  = child_bm25.get_scores(tokenize(question))
    bm25_ranking = list(bm25_scores.argsort()[::-1][:first_stage_k])

    # ── Stage 1b: vector search over child chunks ─────────────────────────
    vec_results  = pc_vs.similarity_search(question, k=first_stage_k)
    c_text_to_idx = {t: i for i, t in enumerate(child_texts)}
    vec_ranking  = [
        c_text_to_idx[doc.page_content]
        for doc in vec_results
        if doc.page_content in c_text_to_idx
    ]

    # ── Stage 1c: RRF on child indices ────────────────────────────────────
    fused = rrf_score([bm25_ranking, vec_ranking], k=60)
    top_child_idxs = sorted(fused, key=fused.get, reverse=True)[:first_stage_k]

    # ── Stage 2: look up parent for each child (de-duplicate) ─────────────
    seen = {}
    for ci in top_child_idxs:
        meta   = child_docs[ci].metadata
        p_idx  = meta['parent_idx']
        cid    = meta['child_id']
        if p_idx not in seen:
            seen[p_idx] = parent_store[cid]

    parent_texts = list(seen.values())

    # ── Stage 3: cross-encoder rerank parents ─────────────────────────────
    if len(parent_texts) > 1:
        ce_pairs  = [(question, t) for t in parent_texts]
        ce_scores = cross_encoder.predict(ce_pairs)
        ranked    = sorted(zip(parent_texts, ce_scores),
                           key=lambda x: x[1], reverse=True)
        top_contexts = [t for t, _ in ranked[:final_k]]
    else:
        top_contexts = parent_texts[:final_k]

    # ── Stage 4: generate ─────────────────────────────────────────────────
    context_block = '\n\n'.join(
        f'[Source {i+1}]\n{c}' for i, c in enumerate(top_contexts)
    )
    answer = ollama_chat([
        {'role': 'system', 'content': RAG_SYSTEM},
        {'role': 'user',   'content': f'CONTEXT:\n{context_block}\n\nQUESTION: {question}'},
    ])
    return {'answer': answer, 'contexts': top_contexts}


# ── Run improved pipeline on test set ─────────────────────────────────────
print('Running 15 questions through improved pipeline...')
print('(May take 5–15 minutes)\n')

improved_results = []
for i, item in enumerate(TEST_SET, 1):
    result = improved_rag(item['question'])
    improved_results.append({
        'question':    item['question'],
        'ground_truth': item['ground_truth'],
        'answer':      result['answer'],
        'contexts':    result['contexts'],
    })
    print(f'  [{i:02d}/15] {item["question"][:60]}...')

print('\nImproved pipeline inference complete.')

In [ ]:
# ── RAGAS on improved pipeline ────────────────────────────────────────────
improved_dataset = Dataset.from_dict({
    'question':    [r['question']     for r in improved_results],
    'answer':      [r['answer']       for r in improved_results],
    'contexts':    [r['contexts']     for r in improved_results],
    'ground_truth': [r['ground_truth'] for r in improved_results],
})

print('Running RAGAS evaluation on improved pipeline...')
improved_eval = evaluate(
    dataset=improved_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_emb,
    raise_exceptions=False,
)

improved_scores = {
    'faithfulness':     round(float(improved_eval['faithfulness']), 3),
    'answer_relevancy': round(float(improved_eval['answer_relevancy']), 3),
    'context_recall':   round(float(improved_eval['context_recall']), 3),
}

# ── Comparison table ──────────────────────────────────────────────────────
print('\n' + '=' * 65)
print(f'{"Metric":<22} {"Baseline":>10} {"Improved":>10} {"Delta":>10}')
print('─' * 65)
for metric in ['faithfulness', 'answer_relevancy', 'context_recall']:
    b = baseline_scores[metric]
    i = improved_scores[metric]
    delta = i - b
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '─')
    print(f'{metric:<22} {b:>10.3f} {i:>10.3f} {arrow} {abs(delta):>7.3f}')
print('─' * 65)

# Which technique helped most?
deltas = {
    metric: improved_scores[metric] - baseline_scores[metric]
    for metric in baseline_scores
}
best_metric = max(deltas, key=deltas.get)
print(f'\nBiggest improvement: {best_metric} (+{deltas[best_metric]:.3f})')
print('\nbaseline_scores  =', baseline_scores)
print('improved_scores  =', improved_scores)

---

## ✅ Summary — Advanced RAG Techniques

### What you built

| Technique | What it fixes | Implementation |
|-----------|--------------|---------------|
| **RAGAS measurement** | "Looks good" → objective numbers | `evaluate()` with local Ollama judge |
| **Parent-child chunking** | Small chunks lose context | child_size=150, parent_size=300, look up parent on retrieval |
| **Cross-encoder reranking** | Bi-encoder misses subtle relevance | Retrieve 20 → rerank with `ms-marco-MiniLM-L-6-v2` → return top 5 |
| **Hybrid BM25 + vector** | Vector search misses exact keywords | RRF fusion (k=60) of BM25 and cosine rankings |

### The improved pipeline architecture

```
User query
    │
    ├──► BM25 (keyword)  ─────────────┐
    │                                 ├── RRF fusion → top-30 child chunks
    ├──► Vector (semantic) ───────────┘        │
    │                                          ▼
    │                              Look up parent chunks
    │                                          │
    │                                          ▼
    │                              Cross-encoder rerank → top 5
    │                                          │
    │                                          ▼
    └────────────────────────────► Ollama generation → grounded answer
```

### Key lessons

1. **Measure first** — never optimise without a numeric baseline. RAGAS gives you that.
2. **Chunking is retrieval** — the quality of your index determines the ceiling of your answers.
3. **Reranking is cheap** — a 22 MB cross-encoder run at query time beats a bigger bi-encoder.
4. **Hybrid is robust** — combining lexical and semantic search covers both exact and fuzzy queries.
5. **All free, all local** — every technique here uses open-source models. No API costs.

### What comes next — Phase 5: Agents Use RAG as a Tool

You now have a high-quality RAG retriever. In Phase 05 you will wrap it as a **tool**
that an LLM agent can call:

| Phase 05 topic | Description |
|----------------|-------------|
| **Tool-use agents** | LLM decides when to call `rag_search(query)` vs other tools |
| **ReAct pattern** | Reason → Act → Observe → Reason loop |
| **Multi-tool agents** | Agent chooses between RAG, calculator, web search, code executor |
| **Memory** | Agent maintains context across sessions using the vector store |

The pipeline you built in Phases 03–04 becomes the knowledge backbone of the agent.
The agent doesn't retrieve — it *decides whether* to retrieve, *what* to retrieve,
and *how* to use the result.